# Import

In [ ]:
import os
from datetime import datetime
from utils import get_datalake_logger
from scripts.mount import ensure_mount
from scripts.write_df_to_parquet import write_df_to_parquet
from scripts.utils_gold import *

# Config

In [ ]:
STORAGE_ACCOUNT_NAME = os.environ["STORAGE_ACCOUNT_NAME"]
FILESYSTEM_NAME_GOLD = os.environ["CONTAINER_GOLD"]
SECRET_SCOPE_NAME = os.environ["SECRET_SCOPE_NAME"]
SECRET_KEY_NAME = os.environ["SECRET_KEY_NAME"]
MOUNT_POINT_GOLD = "/mnt/donnees-qualite-eau-gold"

# Configuration du logging

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = f"notebooks/04_aggregate_data_to_gold/run_{timestamp}.log"
logger = get_datalake_logger(
    logger_name="04_aggregate_data_to_gold",
    account_name=STORAGE_ACCOUNT_NAME,
    account_key=dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=SECRET_KEY_NAME),
    filesystem_name="logs",
    log_path=log_path
)

# Vérifier / Créer le montage du Data Lake si nécessaire

In [ ]:
# Crée les montages si besoin
ensure_mount(
    mount_point=MOUNT_POINT_GOLD,
    container_name=FILESYSTEM_NAME_GOLD,
    secret_scope_name=SECRET_SCOPE_NAME,
    secret_key_name=SECRET_KEY_NAME,
    storage_account_name=STORAGE_ACCOUNT_NAME,
    logger=logger
)

# Config Schema

In [ ]:
logger.info("Vérification de l’existence du schéma 'gold'...")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
logger.info("Le schéma 'gold' est prêt à être utilisé.")

# Récupération des données du schema Silver

In [ ]:
df_plv_silver = spark.table("silver.dis_plv")
df_result_silver = spark.table("silver.dis_result")
df_com_silver = spark.table("silver.dis_com")

# Creation des table de dimension

## Info reseaux et communes

In [ ]:
# Table des communes
window_spec = Window.partitionBy("insee_commune").orderBy(F.col("annee").desc())

dim_commune = (
    df_com_silver
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .select("insee_commune", "nom_commune", "annee")
).drop("annee")

In [ ]:
# Table des réseaux
window_spec_reseau = Window.partitionBy("cd_reseau").orderBy(F.col("annee").desc())

dim_reseau = (
    df_com_silver
    .withColumn("row_number", F.row_number().over(window_spec_reseau))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .select("cd_reseau", "nom_reseau", "debut_alim", "annee")
).drop("annee")

In [ ]:
# Table de liaison : commune ↔ réseau ↔ quartier
dim_affectation_plv = df_com_silver.select(
    "insee_commune", "cd_reseau", "quartier"
).distinct()

In [ ]:
window_param = Window.partitionBy("cd_parametre").orderBy(F.col("annee").desc())
# Table des infos des paramètres 
dim_parametre_info = (
    df_result_silver
    .withColumn("row_number", F.row_number().over(window_param))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .select(
        "cd_parametre",
        "lib_parametre",
        "cd_parametre_sise_eaux",
        "is_qualitatif"
    )
)

In [ ]:
# Table des unités
dim_unite = (
    df_result_silver
    .withColumn("row_number", F.row_number().over(window_param))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .select(
        "cd_parametre",
        "cd_unite_reference_sise_eaux",
        "cd_unite_reference",
    )
)

In [ ]:
# Table des valeurs de référence
dim_valeur_ref = (
    df_result_silver
    .withColumn("row_number", F.row_number().over(window_param))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .select(
        "cd_parametre",
        "min_val_ref",
        "max_val_ref",
        "valeur_limite",
    )
)